# Частина 2
## Завдання №1 
Завантажити та відкрити датасет: Individual Household Electric Power Consumption Dataset.Датасет знаходиться локально у файлі household_power_consumption.txt.


In [11]:
import pandas as pd
import timeit
def load_power_consumption(file_path):
    df=pd.read_csv(file_path,sep=';',na_values=['?'])
    return df
start=timeit.default_timer()
df=load_power_consumption('household_power_consumption.txt')
end=timeit.default_timer()
print("Час завантаження:", end-start, "секунд")
print(df.info())

Час завантаження: 1.5423268000013195 секунд
<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   str    
 1   Time                   str    
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), str(2)
memory usage: 142.5 MB
None


 ## Завдання №2
Здійснити data cleaning.

In [3]:
def clean_data(df):
    df_clean=df.dropna()
    return df_clean
start_load=timeit.default_timer()
df=load_power_consumption('household_power_consumption.txt')
end_load=timeit.default_timer()
print("Час завантаження:",end_load-start_load,"секунд")
start_clean=timeit.default_timer()
df_clean=clean_data(df)
end_clean=timeit.default_timer()
print("Час очищення:",end_clean-start_clean,"секунд")
print(df_clean.head())

Час завантаження: 1.5070666999963578 секунд
Час очищення: 0.2521814999927301 секунд
         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  


## Завдання № 3.1
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [4]:
def get_power(df,power=5):
    return df[df['Global_active_power']>power]
start=timeit.default_timer()
high_power=get_power(df_clean, 5)
end=timeit.default_timer()
print("Час:",end-start,"секунд")
print(high_power[['Global_active_power','Date','Time']].head())

Час: 0.011491700017359108 секунд
    Global_active_power        Date      Time
1                 5.360  16/12/2006  17:25:00
2                 5.374  16/12/2006  17:26:00
3                 5.388  16/12/2006  17:27:00
11                5.412  16/12/2006  17:35:00
12                5.224  16/12/2006  17:36:00


## Завдання № 3.2
Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильник (Sub_metering_2) споживають більше, ніж бойлер та кондиціонер (Sub_metering_3).

In [5]:
def analyze_appliances(df):
    current_filter=(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)
    df_filtered=df[current_filter]
    result=df_filtered[df_filtered['Sub_metering_2'] > df_filtered['Sub_metering_3']]
    return result
start=timeit.default_timer()
result=analyze_appliances(df_clean)
end=timeit.default_timer()
print("Час:",end-start,"секунд")
print(f"Знайдено:{len(result)} записів")
print(result[['Global_intensity', 'Sub_metering_2', 'Sub_metering_3']].head())

Час: 0.013526299997465685 секунд
Знайдено:2509 записів
     Global_intensity  Sub_metering_2  Sub_metering_3
45               19.0            37.0            16.0
460              19.6            13.0             0.0
464              19.6            27.0             0.0
475              19.4            36.0             0.0
476              19.4            35.0             0.0


## Завдання № 3.3
Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [6]:
def sample_average(df,n=500000):
    start=timeit.default_timer()
    sample=df.sample(n=n)
    means=sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    end=timeit.default_timer()
    print(f"Час виконання:{end-start:.4f} секунд")
    print(f"Розмір вибірки:{n}")
    print("Середні:")
    print(means)
    return sample,means
sample,means=sample_average(df_clean,500000)

Час виконання:0.2392 секунд
Розмір вибірки:500000
Середні:
Sub_metering_1    1.112114
Sub_metering_2    1.298056
Sub_metering_3    6.443730
dtype: float64


## Завдання № 3.4
Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [7]:
def smart_sample(df):
    evening=df[df['Time']>='18:00:00']
    high_power=evening[evening['Global_active_power']>6]
    final=high_power[(high_power['Sub_metering_2']>high_power['Sub_metering_1']) & 
                       (high_power['Sub_metering_2']>high_power['Sub_metering_3'])]
    if len(final)>=2:
        mid=len(final)//2
        first=final.iloc[:mid:3]
        second=final.iloc[mid::4]
        final=pd.concat([first, second])
    return final
start=timeit.default_timer()
result=smart_sample(df_clean)
end=timeit.default_timer()
print("Результат вибірки:")
print(result[['Time', 'Global_active_power', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].head())
print(f"\n Час виконання: {end - start:.4f} секунд")

Результат вибірки:
           Time  Global_active_power  Sub_metering_1  Sub_metering_2  \
41     18:05:00                6.052             0.0            37.0   
44     18:08:00                6.308             0.0            36.0   
17494  20:58:00                6.386             1.0            36.0   
17498  21:02:00                8.088             1.0            72.0   
17501  21:05:00                7.230             1.0            73.0   

       Sub_metering_3  
41               17.0  
44               17.0  
17494            17.0  
17498            17.0  
17501            17.0  

 Час виконання: 0.2035 секунд


## Завдання № 4
Пронормувати та стандартизувати вибраний датасет.Використовуються MinMaxScaler та StandardScaler

In [9]:
!pip install scikit-learn

In [15]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
def normalize_data(df, cols):
    data=df[cols].copy()
    scaler=MinMaxScaler()
    norm_data=scaler.fit_transform(data)
    return pd.DataFrame(norm_data, columns=cols)
def standardize_data(df, cols):
    data=df[cols].copy()
    scaler=StandardScaler()
    std_data=scaler.fit_transform(data)
    return pd.DataFrame(std_data, columns=cols)
cols=['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity']
df_norm=normalize_data(df, cols)
df_std=standardize_data(df, cols)
print("Оригінал:")
print(df[cols].head(3))
print("\nНормовані [0,1]:")
print(df_norm.head(3))
print("\nСтандартизовані (сер=0, дисперсія=1):")
print(df_std.head(3))

Оригінал:
   Global_active_power  Global_reactive_power  Voltage  Global_intensity
0                4.216                  0.418   234.84              18.4
1                5.360                  0.436   233.63              23.0
2                5.374                  0.498   233.29              23.0

Нормовані [0,1]:
   Global_active_power  Global_reactive_power   Voltage  Global_intensity
0             0.374796               0.300719  0.376090          0.377593
1             0.478363               0.313669  0.336995          0.473029
2             0.479631               0.358273  0.326010          0.473029

Стандартизовані (сер=0, дисперсія=1):
   Global_active_power  Global_reactive_power   Voltage  Global_intensity
0             2.955077               2.610721 -1.851816          3.098789
1             4.037085               2.770406 -2.225274          4.133800
2             4.050326               3.320432 -2.330213          4.133800


## Завдання № 5
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів. Обрано колонки (Global_active_power) та (Global_intensity).

In [18]:
def calculate_coefficient(df, col1='Global_active_power', col2='Global_intensity'):
    pearson=df[col1].corr(df[col2], method='pearson')
    spearman=df[col1].corr(df[col2], method='spearman')
    print(f"Коефіцієнт кореляції Пірсона ({col1} та {col2}):")
    print(f"{pearson:.4f}")
    print(f"\n Коефіцієнт кореляції Спірмена ({col1} та {col2}):")
    print(f"{spearman:.4f}")
calculate_coefficient(df)

Коефіцієнт кореляції Пірсона (Global_active_power та Global_intensity):
0.9989

 Коефіцієнт кореляції Спірмена (Global_active_power та Global_intensity):
0.9954


## Завдання № 6
Провести One Hot Encoding категоріального атрибута.Створюю категорії часу(ніч,ранок,день,вечір).


In [19]:
def create_time_category(df):
    df=df.copy()
    df['Hour']=pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour
    def period(h):
        if h<6:return 'ніч'
        if h<12:return 'ранок'
        if h<18:return 'день'
        return 'вечір'
    df['Time_of_Day']=df['Hour'].apply(period)
    return df
df_cat=create_time_category(df_clean)
one_hot=pd.get_dummies(df_cat['Time_of_Day'], prefix='час')
df_final=pd.concat([df_cat[['Time', 'Time_of_Day']], one_hot], axis=1)
print(df_final.head(100))

        Time Time_of_Day  час_вечір  час_день  час_ніч  час_ранок
0   17:24:00        день      False      True    False      False
1   17:25:00        день      False      True    False      False
2   17:26:00        день      False      True    False      False
3   17:27:00        день      False      True    False      False
4   17:28:00        день      False      True    False      False
..       ...         ...        ...       ...      ...        ...
95  18:59:00       вечір       True     False    False      False
96  19:00:00       вечір       True     False    False      False
97  19:01:00       вечір       True     False    False      False
98  19:02:00       вечір       True     False    False      False
99  19:03:00       вечір       True     False    False      False

[100 rows x 6 columns]
